In [1]:
using DifferentialEquations # for the actual time evolution
using OrdinaryDiffEq # for ODEs
using Plots # for plotting
using Base.Threads # for parallelization
using StaticArrays # somehow needed to use multiple variables in DifferentialEquations.jl

using Plots, LaTeXStrings, Colors
using Plots.PlotMeasures
using LinearAlgebra

using Random, Distributions

using FFTW # discrete Fourier transform

using JLD2 # for file saving

In [2]:
level = "../../../../"

include(joinpath(level, "src/4th-order-FD-stencils.jl"));
include(joinpath(level, "src/evolution_Liouville_larger_cutoff.jl"));
include(joinpath(level, "src/hamiltonian_Liouville.jl"));
include(joinpath(level, "src/initial_data_waves.jl"));
include(joinpath(level, "src/visualisation.jl"));

### evolution

In [3]:
function artisan_evolution_at_resolution(Nx, stableRandomSeed, pModel, pInit, target_time)
    # unpack model parameters
    (mphi2, mchi2, c4, c, epsDiss) = pModel;
    
    # set the spatial discretization
    NboundaryPadding = 2;#Int(div(Nx,2));  # Number of boundary padding points
    dx = 1/(Nx);  # Grid spacing
        
    pGrid = (dx, Nx, NboundaryPadding);
    # reset a combined set of parameters (residual from old structure ... could be modified)
    # TODO: modify to pGrid, pModel, pInit
    p = (dx, mphi2, mchi2, c4, c, epsDiss, Nx, NboundaryPadding);

    # set the time span
    tspan = (0, target_time);

    # set the evolution method
    time_integration_method = RK4();
    
    # generate initial conditions
    u0 = initial_data(
        range(0, step=dx, length=(Nx + 2 * NboundaryPadding)),
        p, 
        pInit
    );
        
    # set the problem
    prob = ODEProblem(finite_differenced_pde_with_bc!, u0, tspan, p);

    sol = solve(
        prob, time_integration_method, 
        saveat = tspan[end]/10^3, #exp.(range(log(tspan[1]), log(tspan[end]), length=10^4))
        dt=dx/10, 
        adaptive = false, 
        dense=false, 
        maxiters=typemax(Int),
        callback=field_size_callback
    );
        
    # obtain the hamiltonian
    hamiltonian = zeros(length(sol.u))
    hamphi = zeros(length(sol.u))
    hamchi = zeros(length(sol.u))
    for i = 1:length(sol.u)
        hamiltonian[i] = nintegrate_simps(hamiltonian_density(sol.u[i], p), dx)
        hamphi[i] = nintegrate_simps(hamiltonian_phi(sol.u[i], p), dx)
        hamchi[i] = nintegrate_simps(hamiltonian_chi(sol.u[i], p), dx)
    end
    
    return (p, sol, hamiltonian, hamphi, hamchi)
end

artisan_evolution_at_resolution (generic function with 1 method)

In [4]:
function evolution_at_param(param, current_target_time, current_res_log2, stableRandomSeed)
    
    #stableRandomSeed = 42
    print("persistent random seed: ", stableRandomSeed, "\n")
    
    print("current ghostly coupling: ", param, "\n")
    
    # set monitoring flags
    convergence_maintained = false;
    lower_bound_only = true;
    
    # parameters of the model
    mphi2 = 1.;
    mchi2 = 1.;
    c4 = param;
    c = -1.;
    epsDiss = 0;

    # parameters of the initial data
    a0phi = 0; # effectively sets the relative amplitude to the stochastic ID (since Tkin is kept fixed)
    a0chi = a0phi; # effectively sets the relative amplitude to the stochastic ID (since Tkin is kept fixed)
    k0phi = 1;
    k0chi = 2 * k0phi;
    x0phi = 0;
    x0chi = 1/3;

    offsetphi = 0;
    offsetchi = 0;

    aStochastic = 4;
    mink = 1;
    maxk = 4;
    
    desiredTkinPhi = NaN;
    desiredTkinChi = NaN;
    
    # set the combined set of parameters 
    pModel = (mphi2, mchi2, c4, c, epsDiss);
    pInit = (
        a0phi, a0chi, k0phi, k0chi, x0phi, x0chi, 
        offsetphi, offsetchi, 
        aStochastic, mink, maxk, stableRandomSeed,
        desiredTkinPhi, desiredTkinChi
    );
     
    # set some tables to store intermediate output
    resTab = [2^i for i in current_res_log2-2:current_res_log2]
    pTab = []
    solTab = []
    hamiltonianTab = []
    hamPhiTab = []
    hamChiTab = []
    
    #############################
    # evolution
    #############################
    
    # run evolution
    for res in resTab
        print("current resolution: ", res, "\n")
        # run the evolution
        @time (p, sol, hamiltonian, hamPhi, hamChi) = artisan_evolution_at_resolution(
            res, stableRandomSeed, pModel, pInit, current_target_time
        )
        print("... terminated", "\n")
        # unpack parameters
        (dx, mphi2, mchi2, c4, c, epsDiss, Nx, NboundaryPadding) = p
        # append the results
        push!(pTab, p)
        push!(solTab, sol)
        push!(hamiltonianTab, hamiltonian)
        push!(hamPhiTab, hamPhi)
        push!(hamChiTab, hamChi)
    end
    
    #############################
    # CONVERGENCE
    #############################
    
    dir_path = string("plots/",stableRandomSeed,"/",param)

    # create the directory if it does not yet exist
    if !isdir(dir_path)
        print("Output plot directory does not exist. Creating it ...\n")
        mkpath(dir_path)
    else
        print("Output plot directory already exists.\n")
    end
    
    # plot and determine convergence 
    loss_of_convergence_time = save_convergence_plots(
        resTab, pTab, solTab, 
        hamiltonianTab, 
        dir_path
    )
    if loss_of_convergence_time >= solTab[end].t[end]
        print("Convergence kept at all times.\n")
    else
        print("Convergence lost at time t=",loss_of_convergence_time,"\n")
    end
    
    # determine the index of convergence loss
    loss_of_convergence_index = findfirst(t -> t > loss_of_convergence_time, solTab[end].t)
    if loss_of_convergence_index === nothing
        loss_of_convergence_index = length(solTab[end].t)
    end
    
    #print(10 * hamPhiTab[end][1],"\n")
    #print(hamPhiTab[end][2:10],"\n")
    
    # determine the onset time of the runaway (10-fold increase in either kinetic energy)
    runaway_index_phi = findfirst(energy -> abs(energy) > 10 * abs(hamPhiTab[end][1]), hamPhiTab[end])
    if runaway_index_phi === nothing
        runaway_index_phi = length(hamPhiTab[end])
    end
    runaway_index_chi = findfirst(energy -> abs(energy) > 10 * abs(hamChiTab[end][1]), hamChiTab[end])
    if runaway_index_chi === nothing
        runaway_index_chi = length(hamChiTab[end])
    end
    runaway_index = min(runaway_index_phi, runaway_index_chi)
    runaway_time = solTab[end].t[runaway_index]
    if runaway_time >= solTab[end].t[end]
        print("No runaway detected.\n")
    else
        print("Runaway detected at time t=",runaway_time,"\n")
    end    
    
    #############################
    # GENERATE REMAINING PLOTS IF DESIRED
    #############################
    
    dir_path = string("plots/",stableRandomSeed,"/",param)

    # create the directory if it does not yet exist
    if !isdir(dir_path)
        #print("Directory does not exist. Creating it...")
        mkpath(dir_path)
    end
        
    # plot energy components
    save_energies_plot(
        resTab, pTab, solTab, 
        hamiltonianTab, hamPhiTab, hamChiTab, 
        dir_path,
        #loss_of_convergence_time=loss_of_convergence_time
    )
    save_normalised_energies_plot(
        resTab, pTab, solTab, 
        hamiltonianTab, hamPhiTab, hamChiTab, 
        dir_path,
        #loss_of_convergence_time=loss_of_convergence_time
    )
    save_difference_in_energies_plot(
        resTab, pTab, solTab, 
        hamiltonianTab, hamPhiTab, hamChiTab, 
        dir_path,
        #loss_of_convergence_time=loss_of_convergence_time
    )
    
    # plot field heatmaps
    save_density_plots(
        solTab[end], pTab[end], pInit,
        dir_path,
        loss_of_convergence_time=loss_of_convergence_time
    )
    
    # save snapshots
    save_snaps(
        solTab[end], pTab[end];
        snap_intervals=Int(round(length(solTab[end])/1)), 
        yrangeVal=1.2,
        dir_path = dir_path
    );
    
    # animate the fields
    save_animation(
        solTab[end][1:max(1,div(loss_of_convergence_index,10^2)):loss_of_convergence_index], 
        pTab[end],
        join([dir_path, "/animation_Nx=", resTab[end], ".gif"])
    );    
#     # animate frequencies
#     save_animation_momentum_space(
#         solTab[end][1:max(1,div(loss_of_convergence_index,10^2)):loss_of_convergence_index], 
#         pTab[end],
#         join([dir_path, "/animation_momentum_space_Nx=", resTab[end], ".gif"])
#     );
    
    print("Finished plotting.", "\n")
    
    #############################
    # SAVE DATA
    #############################
    
    if runaway_time > loss_of_convergence_time
        print("WARNING: Convergence not maintained until onset of runaway. Resolution insufficient.", "\n")
    else
        convergence_maintained = true
        if runaway_time >= solTab[end].t[end]
            print("WARNING: Lower bound only because target time insufficient.", "\n")
        else
            lower_bound_only = false
        end
    end
    
    dir_path = string("dat/",stableRandomSeed)
    if !isdir(dir_path)
        mkpath(dir_path)
    end

    timesteps = solTab[end].t
    stable_until = min(runaway_time, loss_of_convergence_time)

    @save joinpath(pwd(), dir_path, string(param,".jld2")) param stable_until lower_bound_only timesteps hamiltonianTab hamPhiTab hamChiTab

    print("Saved data.", "\n")

    
    return (runaway_time, convergence_maintained, lower_bound_only)
end

evolution_at_param (generic function with 1 method)

### main()

In [14]:
#param_base = 1.2
#param_table = reverse([param_base^i for i in -16:12])

param_table = [invC4 for invC4 in 1:8:128]
param_table = param_table.^(-1)

16-element Vector{Float64}:
 1.0
 0.2
 0.1111111111111111
 0.07692307692307693
 0.058823529411764705
 0.047619047619047616
 0.04
 0.034482758620689655
 0.030303030303030304
 0.02702702702702703
 0.024390243902439025
 0.022222222222222223
 0.02040816326530612
 0.018867924528301886
 0.017543859649122806
 0.01639344262295082

In [15]:
function main()
    
    # some random seed (can be modified at will)
    stableRandomSeed = rand(1:10^7)
    
    # initialise flags
    convergence_maintained = false;
    lower_bound_only = true;
    
    # set abort criteria ...
    highest_res_log2 = 12;
    max_target_time = 2 * 10^3;
    # ... and their initial values
    current_res_log2 = 10;
    current_target_time = 1;
    
    # initialise the runaway time for handover to next param value
    runaway_time = Inf;
    
    # set table of desired param_table (NOTE: links to scaling assumption below)
    param_base = 1
    param_table = [invC4 for invC4 in 1:8:128]
    param_table = param_table.^(-1)
    
    # loop over all values in param_table
    for param in param_table
        
        # re-attempt while flags not positive or until abort criteria met
        while (!convergence_maintained||lower_bound_only) && (current_res_log2 <= highest_res_log2) && (current_target_time <= max_target_time)
            # attempt run and obtain flags
            (runaway_time, convergence_maintained, lower_bound_only) = evolution_at_param(
                param, 
                current_target_time, 
                current_res_log2,
                stableRandomSeed
            )
            # update according to obtained flags
            if convergence_maintained                
                if lower_bound_only
                    current_target_time = current_target_time * 4
                    print("Increasing target time to T = ", current_target_time, "\n")
                else
                    current_target_time = min(runaway_time, current_target_time);
                    print("Target time reset to confidently detected runaway time T = ", current_target_time, "\n")
                end
            else
                current_res_log2 = current_res_log2 + 1;
                print("Increasing resolution from N = ", current_res_log2 - 1, " to ", current_res_log2, "\n")
            end
        end
        
        print("PARAM = ", param, " DONE!\n")
        
        # update target time based on the presumed scaling assumption and adapt the target time accordingly
        current_target_time = runaway_time
        print("Updating target time for next param value from T = ", current_target_time, " ... ")
        current_target_time = current_target_time * exp(param_base)     
        print("to T = ", current_target_time, "\n")
        
        # decrease resolution if convergence was maintained in previous step
#         if convergence_maintained
#             current_res_log2 = current_res_log2 - 1;
#             print("Decreasing resolution from N = ", current_res_log2 + 1, " to ", current_res_log2, "\n")
#         end
        
        # check whether it makes sense to go on; otherwise abort 
        if convergence_maintained && lower_bound_only && current_target_time >= max_target_time
            print("ABORT: maximum target time approached in converged simulation; no use to proceed")
            return
        end
        
        # ensure that current params don't exceed the abort criteria for the next step
        current_res_log2 = min(current_res_log2, highest_res_log2)
        current_target_time = min(current_target_time, max_target_time)
        
        # reset the flags
        convergence_maintained = false;
        lower_bound_only = true;
    end

end

main (generic function with 1 method)

In [ ]:
main()

persistent random seed: 1404747
current characteristic frequency: 1.0
current resolution: 512
	user-assigned no rescaling:
		max |amplitude| phi before rescaling: 3.7673929991928716
		max |amplitude| chi before rescaling: 4.007683017284089
  0.699713 seconds (411.11 k allocations: 1.078 GiB, 11.99% gc time)
... terminated
current resolution: 1024
	user-assigned no rescaling:
		max |amplitude| phi before rescaling: 3.7673929991928716
		max |amplitude| chi before rescaling: 4.007683017284089
  2.733251 seconds (779.78 k allocations: 4.025 GiB, 28.92% gc time)
... terminated
current resolution: 2048
	user-assigned no rescaling:
		max |amplitude| phi before rescaling: 3.7673929991928716
		max |amplitude| chi before rescaling: 4.007835872366895
 10.998815 seconds (2.18 M allocations: 15.496 GiB, 13.85% gc time)
... terminated
Output plot directory does not exist. Creating it ...
Convergence kept at all times.
No runaway detected.
Finished plotting.
Saved data.
Increasing target time to T = 

[ Info: Saved animation to /home/aaron/Dropbox/university/0_research/2022-2025/stable_ghosts_Paris/field_theory/calculations_aaron/julia_code/1+1/v6_on_Thanos/00_THANOS_TEMP/L=1_C4=1_m2=1/04_coupling/03_rand/plots/1404747/1.0/animation_Nx=2048.gif


Terminating because one of the fields grew too large at time t = 3.2074218749992225.
  3.265568 seconds (1.22 M allocations: 3.237 GiB, 8.22% gc time)
... terminated
current resolution: 1024
	user-assigned no rescaling:
		max |amplitude| phi before rescaling: 3.7673929991928716
		max |amplitude| chi before rescaling: 4.007683017284089
Terminating because one of the fields grew too large at time t = 3.207226562498153.
  6.725670 seconds (2.40 M allocations: 12.483 GiB, 10.78% gc time)
... terminated
current resolution: 2048
	user-assigned no rescaling:
		max |amplitude| phi before rescaling: 3.7673929991928716
		max |amplitude| chi before rescaling: 4.007835872366895
Terminating because one of the fields grew too large at time t = 3.2072265625031076.
 21.653818 seconds (6.87 M allocations: 48.859 GiB, 14.23% gc time)
... terminated
Output plot directory already exists.
Convergence kept at all times.
Runaway detected at time t=2.568
Finished plotting.
Saved data.
Target time reset to con

[ Info: Saved animation to /home/aaron/Dropbox/university/0_research/2022-2025/stable_ghosts_Paris/field_theory/calculations_aaron/julia_code/1+1/v6_on_Thanos/00_THANOS_TEMP/L=1_C4=1_m2=1/04_coupling/03_rand/plots/1404747/1.0/animation_Nx=2048.gif


  7.103498 seconds (2.62 M allocations: 6.977 GiB, 8.39% gc time)
... terminated
current resolution: 1024
	user-assigned no rescaling:
		max |amplitude| phi before rescaling: 3.7673929991928716
		max |amplitude| chi before rescaling: 4.007683017284089
 22.291153 seconds (5.19 M allocations: 27.036 GiB, 10.96% gc time)
... terminated
current resolution: 2048
	user-assigned no rescaling:
		max |amplitude| phi before rescaling: 3.7673929991928716
		max |amplitude| chi before rescaling: 4.007835872366895
 81.014637 seconds (14.92 M allocations: 106.080 GiB, 8.08% gc time)
... terminated
Output plot directory does not exist. Creating it ...
Convergence kept at all times.
No runaway detected.
Finished plotting.
Saved data.
Increasing target time to T = 27.922190941931312
persistent random seed: 1404747
current characteristic frequency: 0.2
current resolution: 512
	user-assigned no rescaling:
		max |amplitude| phi before rescaling: 3.7673929991928716
		max |amplitude| chi before rescaling: 4.

...

Terminating because one of the fields grew too large at time t = 47.28339843735977.
 41.803440 seconds (17.45 M allocations: 46.687 GiB, 7.82% gc time)
... terminated
current resolution: 1024
	user-assigned no rescaling:
		max |amplitude| phi before rescaling: 3.7673929991928716
		max |amplitude| chi before rescaling: 4.007683017284089
Terminating because one of the fields grew too large at time t = 42.61328124965382.
106.251813 seconds (31.44 M allocations: 164.048 GiB, 8.11% gc time)
... terminated
current resolution: 2048
	user-assigned no rescaling:
		max |amplitude| phi before rescaling: 3.7673929991928716
		max |amplitude| chi before rescaling: 4.007835872366895
Terminating because one of the fields grew too large at time t = 42.434179687778645.
408.094589 seconds (90.42 M allocations: 642.899 GiB, 8.17% gc time)
... terminated
Output plot directory does not exist. Creating it ...
Convergence lost at time t=36.44974629004963
Runaway detected at time t=31.280190926797403
Finished 

[ Info: Saved animation to /home/aaron/Dropbox/university/0_research/2022-2025/stable_ghosts_Paris/field_theory/calculations_aaron/julia_code/1+1/v6_on_Thanos/00_THANOS_TEMP/L=1_C4=1_m2=1/04_coupling/03_rand_A=4/plots/1404747/0.02702702702702703/animation_Nx=2048.gif


Saved data.
Target time reset to confidently detected runaway time T = 31.280190926797403
PARAM = 0.02702702702702703 DONE!
Updating target time for next param value from T = 31.280190926797403 ... to T = 85.02837458704288
persistent random seed: 1404747
current characteristic frequency: 0.024390243902439025
current resolution: 512
	user-assigned no rescaling:
		max |amplitude| phi before rescaling: 3.7673929991928716
		max |amplitude| chi before rescaling: 4.007683017284089
Terminating because one of the fields grew too large at time t = 40.394726562460015.
 35.539265 seconds (14.91 M allocations: 39.886 GiB, 8.10% gc time)
... terminated
current resolution: 1024
	user-assigned no rescaling:
		max |amplitude| phi before rescaling: 3.7673929991928716
		max |amplitude| chi before rescaling: 4.007683017284089
Terminating because one of the fields grew too large at time t = 40.67744140593199.
118.228007 seconds (30.01 M allocations: 156.599 GiB, 7.71% gc time)
... terminated
current resol

[ Info: Saved animation to /home/aaron/Dropbox/university/0_research/2022-2025/stable_ghosts_Paris/field_theory/calculations_aaron/julia_code/1+1/v6_on_Thanos/00_THANOS_TEMP/L=1_C4=1_m2=1/04_coupling/03_rand/plots/1404747/0.024390243902439025/animation_Nx=2048.gif


Terminating because one of the fields grew too large at time t = 60.69355468716463.
 48.925247 seconds (22.40 M allocations: 59.922 GiB, 8.63% gc time)
... terminated
current resolution: 1024
	user-assigned no rescaling:
		max |amplitude| phi before rescaling: 3.7673929991928716
		max |amplitude| chi before rescaling: 4.007683017284089
Terminating because one of the fields grew too large at time t = 55.303124999469155.
148.242582 seconds (40.80 M allocations: 212.890 GiB, 7.96% gc time)
... terminated
current resolution: 2048
	user-assigned no rescaling:
		max |amplitude| phi before rescaling: 3.7673929991928716
		max |amplitude| chi before rescaling: 4.007835872366895


### export .jl for production run

In [2]:
using NBInclude
nbexport("main.jl", "main.ipynb")